In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.0 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 12.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 11.7 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ultralytics] [ultralytics]n]


In [3]:
import pandas as pd
import os
import shutil
from tqdm import tqdm
from ultralytics import YOLOWorld

# ==========================================
# 1. YOLO-World 모델 및 타겟 클래스 설정
# ==========================================
# 5060Ti에 가볍게 올라가는 'Small' 버전의 YOLO-World 로드
model = YOLOWorld('yolov8s-world.pt')

# 💡 찾고자 하는 재활용품 클래스를 영어로 지정 (YOLO-World는 영어 프롬프트를 사용합니다)
# 필요에 따라 대회 데이터에 맞게 단어를 추가/수정하세요.
custom_classes = [
    "plastic bottle", "glass bottle", "aluminum can", "paper box", 
    "plastic cup", "paper cup", "plastic bag", "styrofoam", "tube container"
]
model.set_classes(custom_classes)

# ==========================================
# 2. YOLO 전용 폴더 구조 자동 생성 함수
# ==========================================
base_dir = './yolo'
dirs_to_make = [
    f'{base_dir}/images/train', f'{base_dir}/images/val',
    f'{base_dir}/labels/train', f'{base_dir}/labels/val'
]

for d in dirs_to_make:
    os.makedirs(d, exist_ok=True)
print("✅ YOLO 폴더 구조 생성 완료 (./yolo/images, ./yolo/labels)")

# ==========================================
# 3. CSV 데이터 읽기 및 Auto-Labeling 실행
# ==========================================
def process_auto_labeling(csv_file, split_name):
    """
    csv_file: 'train_class.csv' 또는 'dev_class.csv'
    split_name: 'train' 또는 'val'
    """
    if not os.path.exists(csv_file):
        print(f"파일 없음 패스: {csv_file}")
        return
        
    df = pd.read_csv(csv_file)
    
    # 우리의 완벽했던 라우터가 'YOLO'로 분류한 행만 필터링!
    yolo_df = df[df['class'] == 'YOLO']
    print(f"\n🚀 [{csv_file}] YOLO 트랙 데이터 {len(yolo_df)}건 라벨링 시작...")
    
    for idx, row in tqdm(yolo_df.iterrows(), total=len(yolo_df)):
        img_path = str(row['path']) # 예: 'train/train_0001.jpg'
        img_name = os.path.basename(img_path)
        
        # 원본 이미지가 존재하는지 확인
        if not os.path.exists(img_path):
            continue
            
        # 1) 이미지 복사 (원본 -> yolo/images/...)
        dest_img_path = os.path.join(base_dir, 'images', split_name, img_name)
        if not os.path.exists(dest_img_path):
            shutil.copy(img_path, dest_img_path)
            
        # 2) YOLO-World 추론 (conf=0.1로 낮춰서 최대한 물체를 많이 잡도록 유도)
        results = model.predict(img_path, conf=0.1, verbose=False)
        
        # 3) Bounding Box 결과를 정답지(.txt) 파일로 직접 작성
        txt_name = img_name.replace('.jpg', '.txt').replace('.png', '.txt')
        dest_txt_path = os.path.join(base_dir, 'labels', split_name, txt_name)
        
        with open(dest_txt_path, 'w') as f:
            for box in results[0].boxes:
                cls_id = int(box.cls[0])
                # xywhn: 중심x, 중심y, 너비, 높이 (0~1 사이 정규화된 비율 값)
                x_center, y_center, width, height = box.xywhn[0]
                f.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

# train_class.csv는 YOLO의 'train' 폴더로, dev_class.csv는 'val' 폴더로 매핑
process_auto_labeling('train_class.csv', 'train')
process_auto_labeling('dev_class.csv', 'val')

# ==========================================
# 4. 학습용 dataset.yaml 파일 자동 생성
# ==========================================
yaml_content = f"""path: {os.path.abspath(base_dir)}
train: images/train
val: images/val

names:
"""
for i, cls_name in enumerate(custom_classes):
    yaml_content += f"  {i}: {cls_name}\n"

yaml_path = os.path.join(base_dir, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)
    
print(f"\n🎉 모든 작업 완료! 학습 설정 파일 생성: {yaml_path}")
print("이제 YOLOv10-S 모델을 학습시킬 완벽한 준비가 끝났습니다.")

✅ YOLO 폴더 구조 생성 완료 (./yolo/images, ./yolo/labels)

🚀 [train_class.csv] YOLO 트랙 데이터 1747건 라벨링 시작...


100% 1747/1747 [01:50<00:00, 15.83it/s]



🚀 [dev_class.csv] YOLO 트랙 데이터 3144건 라벨링 시작...


100% 3144/3144 [03:19<00:00, 15.75it/s]


🎉 모든 작업 완료! 학습 설정 파일 생성: ./yolo/dataset.yaml
이제 YOLOv10-S 모델을 학습시킬 완벽한 준비가 끝났습니다.
